In [ ]:
import torch
import argparse
from tqdm.notebook import tqdm
import cProfile
import os

# Import your new specialized modules
from Adversarial_TTS.model_loader import initialize_environment, load_optimizer
from Adversarial_TTS.core_logic import run_optimization_generation
from Adversarial_TTS.reporting import finalize_run

class NotebookArgs:
    def __init__(self):
        # String parameters
        self.ground_truth_text = "I think the NFL is lame and boring"
        self.target_text = "The Seattle Seahawks are the best Team in the world"

        # Numeric parameters
        self.loop_count = 1
        self.num_generations = 4
        self.pop_size = 4
        self.iv_scalar = 0.5
        self.size_per_phoneme = 1
        self.batch_size = -1

        # Boolean parameters
        self.notify = False
        self.subspace_optimization = False

        # Enum/Selection parameters
        self.mode = "TARGETED"
        self.ACTIVE_OBJECTIVES = ["PESQ", "WHISPER_PROB"]
        self.thresholds = ["PESQ=0.2", "WHISPER_PROB=0.6"]

args = NotebookArgs()

In [ ]:
# 1. Set Device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# 2. Initialize Environment
# This handles Enums, Thresholds, Model Loading, and Reference Data generation
config_data, model_data, audio_data, embedding_data = initialize_environment(args, device)

if config_data is None:
    print("Initialization failed.")
else:
    print(f"Starting Optimization Loop...")

In [ ]:
for iteration in tqdm(range(config_data.loop_count), desc="Total Progress"):
    # Run the generation loop with ObjectiveManager
    fitness_data, progress_bar, stop_optimization, gen = run_optimization_generation(
        config_data=config_data,
        model_data=model_data,
        audio_data=audio_data,
        embedding_data=embedding_data,
        objective_manager=objective_manager,
        iteration=iteration,
        device=device
    )

    # 4. Finalize and save results
    folder_path = finalize_run(
        config_data, fitness_data, model_data, audio_data, progress_bar, gen, device
    )

    # Reset optimizer for next iteration
    model_data.optimizer = load_optimizer(audio_data, config_data)